# CBMS — Activity Classifier Training

Trains a **bidirectional 2-layer GRU** on MediaPipe Pose keypoint sequences extracted from CCTV footage.

### What changed vs. the previous notebook
| # | Change | Why |
|---|--------|-----|
| 1 | LSTM → bidirectional GRU with mean pooling | Matches updated architecture; better gradient flow |
| 2 | Train/val split **before** augmentation | Fixes data leakage that caused fake 100 % val accuracy |
| 3 | Stratified split | Guarantees every class appears in both sets |
| 4 | Dynamic class weights | Computed from actual chunk counts, not hardcoded clip counts |
| 5 | Early stopping (patience = 15) | Stops training when val accuracy plateaus |
| 6 | Architecture tag in checkpoint | Pipeline can assert correct model type on load |
| 7 | Quality filter in `extract_sequences` | Drops windows with >40% zero-vectors (failed pose detection) from training |
| 8 | Speed-jitter augmentation | Subsamples or repeats frames to teach the model pace-invariant gestures |
| 9 | `MINORITY_COPIES` raised 2→3 | Extra augmented copies for sparse activity classes |

### Dataset layout expected
```
/kaggle/input/cbms-training-data/
    spitting/    ← short video clips of spitting
    littering/   ← short video clips of littering
    normal/      ← clips of walking, talking, yawning, drinking …
```
Add more subdirectories to add more classes.

> **Run order:** Cell 1 (install) → restart session → Cells 2 onward.

## Cell 1 — Install dependencies
Run this cell, then **restart the session** before continuing.

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "mediapipe==0.10.14", "opencv-python-headless",
])
print("Done. Restart the session now, then run from Cell 2.")

## Cell 2 — Imports & reproducibility

In [ ]:
import os
import json
import random
import numpy as np
from pathlib import Path
from collections import Counter

import cv2
import mediapipe as mp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## Cell 3 — Configuration
All hyperparameters and paths in one place.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR             = "/kaggle/input/cbms-training-data"
# Kaggle dataset: Screen_Clippings (screen recordings of littering events)
SCREEN_CLIPPINGS_DIR = "/kaggle/input/screen-clippings"
OUTPUT_DIR  = "/kaggle/working/"
MODEL_PATH  = os.path.join(OUTPUT_DIR, "activity_classifier.pt")
LABELS_PATH = os.path.join(OUTPUT_DIR, "class_labels.json")

# ── Sequence ──────────────────────────────────────────────────────────────────
N_FRAMES    = 32        # must match WATCH_WINDOW in the pipeline
N_KEYPOINTS = 33        # MediaPipe full-body landmark count
FEATURES    = N_KEYPOINTS * 3   # 99 floats per frame (x-nose, y-nose, z)
OVERLAP     = 16        # sliding window step = N_FRAMES - OVERLAP

# ── Model ─────────────────────────────────────────────────────────────────────
HIDDEN_1 = 256
HIDDEN_2 = 128
DROPOUT  = 0.3

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS          = 80
BATCH_SIZE      = 16
LR              = 1e-3
VAL_SPLIT       = 0.20
AUGMENT         = True
MINORITY_COPIES = 3     # FIX-5: raised from 2 → 3 to compensate for small dataset
EARLY_STOP_PAT  = 15
LR_PATIENCE     = 8

print("Config loaded.")
print(f"  N_FRAMES={N_FRAMES}  FEATURES={FEATURES}  OVERLAP={OVERLAP}")
print(f"  HIDDEN_1={HIDDEN_1}  HIDDEN_2={HIDDEN_2}  DROPOUT={DROPOUT}")
print(f"  EPOCHS={EPOCHS}  BATCH={BATCH_SIZE}  LR={LR}  VAL_SPLIT={VAL_SPLIT}")
print(f"  EARLY_STOP_PAT={EARLY_STOP_PAT}  LR_PATIENCE={LR_PATIENCE}")
print(f"  MINORITY_COPIES={MINORITY_COPIES}  (raised to help sparse activity classes)")


## Cell 4 — Pose extraction helpers

Keypoints are **nose-relative** (zero-centred): each landmark's `x` and `y` are subtracted by the nose landmark's coordinates before storing. This makes features translation-invariant — a person on the left of the frame produces the same feature vector as the same action performed on the right.
<!-- 
> ⚠️ This preprocessing **must** be replicated exactly in the pipeline's `extract_keypoints()`. Omitting it causes the model to see out-of-distribution input and produce garbage confidences. -->

In [ ]:
# FIX-4 (training side): static_image_mode=True was already correct here.
# This is the reference implementation — inference must mirror this exactly.
_pose = mp.solutions.pose.Pose(
    static_image_mode=True,
    min_detection_confidence=0.3,
)


def _extract_frame_kp(frame: np.ndarray) -> np.ndarray:
    """
    Returns a 99-d nose-relative keypoint vector for one frame,
    or an all-zero vector if MediaPipe finds no landmarks.
    Runs on the FULL frame — do not crop before calling this.
    The pipeline must replicate this exactly (full frame, static mode).
    """
    rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = _pose.process(rgb)
    if result.pose_landmarks:
        kp   = result.pose_landmarks.landmark
        nose = kp[0]
        return np.array(
            [[lm.x - nose.x, lm.y - nose.y, lm.z] for lm in kp],
            dtype=np.float32,
        ).flatten()
    return np.zeros(FEATURES, dtype=np.float32)


def _is_quality_sequence(seq: np.ndarray, max_zero_frac: float = 0.40) -> bool:
    """
    Reject sequences where MediaPipe failed on too many frames.
    A sequence with >40% zero-vectors is mostly noise and will train
    the model to associate zero-vectors with a class label.
    """
    zero_frames = np.all(seq == 0, axis=1).sum()
    return (zero_frames / len(seq)) <= max_zero_frac


def extract_sequences(video_path: str,
                      n_frames: int = N_FRAMES,
                      overlap:  int = OVERLAP) -> list:
    """
    Decode every frame, extract nose-relative pose on the FULL FRAME,
    then slice into overlapping windows of length n_frames.
    Sequences with >40 % missing landmarks are discarded.
    Returns a list of (n_frames, FEATURES) arrays.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return []

    all_kp = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        all_kp.append(_extract_frame_kp(frame))
    cap.release()

    if len(all_kp) < n_frames:
        return []

    step   = n_frames - overlap
    chunks = []
    for start in range(0, len(all_kp) - n_frames + 1, step):
        seq = np.stack(all_kp[start : start + n_frames])
        if _is_quality_sequence(seq):          # FIX-5: drop low-quality windows
            chunks.append(seq)
    return chunks


print("Pose helpers defined.")
print("  static_image_mode=True  |  quality filter: max 40 % zero-frames per window")


## Cell 5 — Load dataset

In [ ]:
def load_dataset(data_dir: str):
    """
    Walk each class subdirectory, extract sliding-window pose sequences.
    Returns X (N, n_frames, FEATURES), y (N,), class_names list.
    """
    data_path   = Path(data_dir)
    class_names = sorted([d.name for d in data_path.iterdir() if d.is_dir()])
    sequences, labels = [], []

    # Explicitly skip the "spitting" class — stripped from this project.
    # If a spitting/ directory exists in the dataset it will be silently ignored.
    EXCLUDED_CLASSES = {"spitting"}
    class_names = [c for c in class_names if c not in EXCLUDED_CLASSES]
    if EXCLUDED_CLASSES:
        print(f"Excluded classes (skipped): {sorted(EXCLUDED_CLASSES)}")

    print(f"Loading dataset from: {data_dir}")
    print(f"Classes found: {class_names}\n")

    for cls_idx, name in enumerate(class_names):
        cls_dir    = data_path / name
        clip_files = (list(cls_dir.glob("*.mp4")) +
                      list(cls_dir.glob("*.avi")) +
                      list(cls_dir.glob("*.mov")))
        n_before = len(sequences)
        ok, skipped = 0, 0
        for clip_path in clip_files:
            chunks = extract_sequences(str(clip_path))
            if not chunks:
                skipped += 1
                continue
            for seq in chunks:
                sequences.append(seq)
                labels.append(cls_idx)
            ok += 1

        n_chunks = len(sequences) - n_before
        print(f"  {name:<15}  {ok:>3} clips  {skipped:>2} skipped  "
              f"→ {n_chunks:>4} chunks")

    if not sequences:
        raise RuntimeError("No sequences loaded. Check DATA_DIR and clip paths.")

    return np.stack(sequences), np.array(labels), class_names


X_raw, y_raw, CLASS_NAMES = load_dataset(DATA_DIR)
print(f"\nRaw dataset: {X_raw.shape}   {dict(Counter(y_raw))}")

## Cell 6 — Stratified train / val split
<!-- 
> **Why split before augmentation?**  
> The previous notebook augmented first, then split. This caused augmented copies of training clips to end up in the validation set, which is why val accuracy showed a misleading 100%. Validation must only ever see raw, unaugmented sequences that the model has never encountered. -->

In [ ]:
def stratified_split(X, y, val_frac, seed):
    """
    Returns (X_train, X_val, y_train, y_val) preserving class ratios.
    Each class contributes proportionally to both sets.
    """
    rng = np.random.default_rng(seed)
    tr_idx, vl_idx = [], []
    for cls in np.unique(y):
        idx   = np.where(y == cls)[0]
        idx   = rng.permutation(idx)
        n_val = max(1, int(len(idx) * val_frac))
        vl_idx.extend(idx[:n_val])
        tr_idx.extend(idx[n_val:])
    return X[tr_idx], X[vl_idx], y[tr_idx], y[vl_idx]


X_train_raw, X_val, y_train_raw, y_val = stratified_split(
    X_raw, y_raw, VAL_SPLIT, SEED
)

print("After stratified split (raw sequences only):")
print(f"  Train  {X_train_raw.shape}   {dict(Counter(y_train_raw))}")
print(f"  Val    {X_val.shape}         {dict(Counter(y_val))}")

## Cell 7 — Augmentation (training set only)

Three transforms applied randomly and independently:
- **Mirror flip** — negate x coordinates (horizontal reflection of the pose)
- **Temporal jitter** — drop 1–2 frames and pad with the last frame, simulating dropped frames or slight speed variation
- **Gaussian noise** — small random perturbation to all keypoint coordinates

Minority classes (non-normal) receive `MINORITY_COPIES` extra augmented copies each to counter class imbalance.

In [ ]:
def augment_sequence(seq: np.ndarray) -> np.ndarray:
    """
    Four transforms applied randomly and independently:
      1. Mirror flip     — negate x coords (horizontal reflection)
      2. Temporal jitter — drop 1-2 frames and pad with last frame
      3. Speed jitter    — subsample (faster) or repeat frames (slower)  [NEW]
      4. Gaussian noise  — small random perturbation
    """
    n, f = seq.shape
    aug  = seq.copy().reshape(n, N_KEYPOINTS, 3)

    # 1. Mirror: negate x (safe — coords are already nose-relative)
    if random.random() > 0.5:
        aug[:, :, 0] = -aug[:, :, 0]

    # 2. Temporal jitter: drop frames and pad at the end
    if random.random() > 0.5:
        skip = random.randint(1, 2)
        kept = sorted(random.sample(range(n), n - skip))
        aug  = np.concatenate([aug[kept],
                               np.tile(aug[kept[-1]], (skip, 1, 1))])

    # 3. Speed jitter: simulate faster (subsample) or slower (repeat) actions
    #    This teaches the model that spitting/littering can happen at varying pace.
    if random.random() > 0.5:
        if random.random() > 0.5:
            # Faster: subsample then repeat last frame to fill n slots
            keep_n   = random.randint(int(n * 0.70), int(n * 0.90))
            indices  = np.linspace(0, n - 1, keep_n, dtype=int)
            subsampled = aug[indices]
            pad_count  = n - keep_n
            aug = np.concatenate([subsampled,
                                   np.tile(subsampled[-1:], (pad_count, 1, 1))])
        else:
            # Slower: repeat a window of frames then trim back to n
            repeat_start = random.randint(0, n // 2)
            repeat_len   = random.randint(2, 5)
            repeated = np.repeat(aug[repeat_start:repeat_start+1], repeat_len, axis=0)
            aug      = np.concatenate([aug[:repeat_start], repeated, aug[repeat_start:]])[:n]

    # 4. Gaussian noise
    aug += np.random.normal(0, 0.005, aug.shape).astype(np.float32)
    return aug.reshape(n, f)


if AUGMENT:
    aug_seqs, aug_labels = [], []
    for seq, label in zip(X_train_raw, y_train_raw):
        aug_seqs.append(augment_sequence(seq))
        aug_labels.append(label)
        if CLASS_NAMES[label] != "normal":
            for _ in range(MINORITY_COPIES):
                aug_seqs.append(augment_sequence(seq))
                aug_labels.append(label)

    X_train = np.concatenate([X_train_raw, np.stack(aug_seqs)])
    y_train = np.concatenate([y_train_raw, np.array(aug_labels)])
else:
    X_train, y_train = X_train_raw.copy(), y_train_raw.copy()

# Shuffle augmented training set
perm    = np.random.permutation(len(X_train))
X_train = X_train[perm]
y_train = y_train[perm]

print("After augmentation:")
print(f"  Train  {X_train.shape}   {dict(Counter(y_train))}")
print(f"  Val    {X_val.shape}     {dict(Counter(y_val))}  (unchanged — no augmentation)")


## Cell 8 — Dynamic class weights

In [ ]:
# Weights are inversely proportional to class frequency in the training set.
# Computed from actual chunk counts after augmentation, not hardcoded clip counts.
counts       = Counter(y_train)
total        = len(y_train)
n_classes    = len(CLASS_NAMES)
class_weights = torch.tensor(
    [total / (n_classes * counts[i]) for i in range(n_classes)],
    dtype=torch.float32,
).to(DEVICE)

print("Class weights (dynamic):")
for i, name in enumerate(CLASS_NAMES):
    print(f"  [{i}] {name:<15}  count={counts[i]:>5}  weight={class_weights[i]:.3f}")

## Cell 9 — Model definition

**Bidirectional GRU** processes the pose sequence in both temporal directions, which is better than a unidirectional LSTM for gesture recognition because the end of the gesture (e.g. the arm returning to rest after littering) is informative about the start. Mean pooling across the time dimension is used instead of last-step output, since the discriminative signal is distributed across the full window rather than concentrated at the final frame.

In [ ]:
class PoseActivityClassifier(nn.Module):
    """
    Bidirectional 2-layer GRU with mean pooling.

    GRU-1   input_size  → hidden1   (bidirectional → output hidden1*2)
    GRU-2   hidden1*2   → hidden2   (bidirectional → output hidden2*2)
    Mean pooling across time dimension
    Dropout
    Linear  hidden2*2   → num_classes
    """
    def __init__(self, input_size, hidden1, hidden2, num_classes, dropout=0.0):
        super().__init__()
        self.gru1       = nn.GRU(input_size,  hidden1, batch_first=True, bidirectional=True)
        self.gru2       = nn.GRU(hidden1 * 2, hidden2, batch_first=True, bidirectional=True)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden2 * 2, num_classes)

    def forward(self, x):
        out, _ = self.gru1(x)
        out, _ = self.gru2(out)
        out    = torch.mean(out, dim=1)     # (batch, hidden2*2)
        return self.classifier(self.dropout(out))


model = PoseActivityClassifier(
    FEATURES, HIDDEN_1, HIDDEN_2, n_classes, DROPOUT
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Bidirectional GRU  |  trainable params: {n_params:,}")
print(model)

## Cell 10 — Data loaders

In [ ]:
X_tr = torch.tensor(X_train, dtype=torch.float32)
y_tr = torch.tensor(y_train, dtype=torch.long)
X_vl = torch.tensor(X_val,   dtype=torch.float32)
y_vl = torch.tensor(y_val,   dtype=torch.long)

train_loader = DataLoader(
    TensorDataset(X_tr, y_tr),
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    TensorDataset(X_vl, y_vl),
    batch_size=BATCH_SIZE,
    pin_memory=torch.cuda.is_available(),
)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")

## Cell 11 — Training loop

Includes:
- **Gradient clipping** at `max_norm=1.0` to prevent exploding gradients
- **`ReduceLROnPlateau`** halves the LR after `LR_PATIENCE` epochs without improvement
- **Early stopping** halts training after `EARLY_STOP_PAT` epochs without a new best val accuracy

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=LR_PATIENCE
)

best_val_acc       = 0.0
best_state         = None
early_stop_counter = 0
history            = {"train_loss": [], "val_acc": [], "lr": []}

print(f"Training — up to {EPOCHS} epochs  (early stop patience={EARLY_STOP_PAT})")
print(f"{'Epoch':>6}  {'Loss':>8}  {'Val acc':>8}  {'LR':>9}")
print("-" * 40)

for epoch in range(1, EPOCHS + 1):

    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(xb)
    avg_loss = total_loss / len(X_train)

    # ── Validate ──────────────────────────────────────────────────────────────
    model.eval()
    correct = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            correct += (model(xb).argmax(dim=1) == yb).sum().item()
    val_acc = correct / len(X_val)

    current_lr = optimizer.param_groups[0]["lr"]
    scheduler.step(val_acc)
    history["train_loss"].append(avg_loss)
    history["val_acc"].append(val_acc)
    history["lr"].append(current_lr)

    # ── Best state & early stopping ───────────────────────────────────────────
    improved = val_acc > best_val_acc
    if improved:
        best_val_acc       = val_acc
        best_state         = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        early_stop_counter = 0
    else:
        early_stop_counter += 1

    if epoch % 5 == 0 or improved:
        marker = "  ← best" if improved else ""
        print(f"{epoch:>6}  {avg_loss:>8.4f}  {val_acc:>7.1%}  "
              f"{current_lr:>9.2e}{marker}")

    if early_stop_counter >= EARLY_STOP_PAT:
        print(f"\nEarly stopping at epoch {epoch} "
              f"— no improvement for {EARLY_STOP_PAT} epochs.")
        break

print(f"\nTraining complete.  Best val accuracy: {best_val_acc:.1%}")

## Cell 12 — Save checkpoint

In [ ]:
model.load_state_dict(best_state)
model.eval()

torch.save(
    {
        "model_state":  best_state,
        "class_names":  CLASS_NAMES,
        "n_frames":     N_FRAMES,
        "features":     FEATURES,
        "hidden1":      HIDDEN_1,
        "hidden2":      HIDDEN_2,
        "val_accuracy": best_val_acc,
        "architecture": "bidirectional_gru",   # asserted by pipeline on load
        "history":      history,
    },
    MODEL_PATH,
)

with open(LABELS_PATH, "w") as f:
    json.dump(CLASS_NAMES, f)

print(f"Model  → {MODEL_PATH}")
print(f"Labels → {LABELS_PATH}")
print(f"Val acc: {best_val_acc:.1%}   Classes: {CLASS_NAMES}")
print()
print(">>> Download both files from the Output tab <<<")

## Cell 13 — Evaluation & confusion matrix

In [ ]:
model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for xb, yb in val_loader:
        probs = F.softmax(model(xb.to(DEVICE)), dim=1).cpu().numpy()
        all_preds.extend(probs.argmax(axis=1))
        all_true.extend(yb.numpy())

all_preds = np.array(all_preds)
all_true  = np.array(all_true)
col_w     = 10

print("Confusion matrix  (rows = actual, cols = predicted):")
print(f"{'':>14}", end="")
for name in CLASS_NAMES:
    print(f"  {name[:col_w]:>{col_w}}", end="")
print()
for i, true_name in enumerate(CLASS_NAMES):
    print(f"  {true_name[:12]:<12}", end="")
    for j in range(n_classes):
        count = ((all_true == i) & (all_preds == j)).sum()
        print(f"  {count:>{col_w}}", end="")
    print()

print()
for i, name in enumerate(CLASS_NAMES):
    mask = all_true == i
    if mask.sum() == 0:
        continue
    acc = (all_preds[mask] == i).mean()
    print(f"  {name:<15}  {acc:.1%}  ({mask.sum()} samples)")

print(f"\n  Overall: {(all_preds == all_true).mean():.1%}")

## Cell 14 — Training curve

In [ ]:
import matplotlib.pyplot as plt

epochs_ran = len(history["train_loss"])
xs = range(1, epochs_ran + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(xs, history["train_loss"], color="steelblue", linewidth=1.5)
ax1.set_title("Training loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.grid(True, alpha=0.3)

ax2.plot(xs, [v * 100 for v in history["val_acc"]], color="seagreen", linewidth=1.5)
ax2.axhline(best_val_acc * 100, color="tomato", linestyle="--",
            linewidth=1, label=f"Best: {best_val_acc:.1%}")
ax2.set_title("Validation accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curve.png"), dpi=120)
plt.show()
print("Saved → training_curve.png")

## Cell 15 — Single-clip inference test

Run this on a known clip to verify the saved checkpoint end-to-end before uploading it to the pipeline dataset.

In [ ]:
def predict_clip(video_path: str, model_path: str) -> dict:
    """
    Load the saved checkpoint and return per-class probabilities averaged
    across all sliding-window chunks extracted from the clip.
    """
    ckpt = torch.load(model_path, map_location="cpu", weights_only=False)

    arch = ckpt.get("architecture", "unknown")
    if arch != "bidirectional_gru":
        print(f"[WARN] Checkpoint architecture tag is '{arch}', "
              f"expected 'bidirectional_gru'. Weights may be incompatible.")

    clf = PoseActivityClassifier(
        ckpt["features"], ckpt["hidden1"], ckpt["hidden2"],
        len(ckpt["class_names"]), dropout=0.0,
    )
    clf.load_state_dict(ckpt["model_state"])
    clf.eval()

    chunks = extract_sequences(video_path, ckpt["n_frames"])
    if not chunks:
        print("No sequences extracted — check the video path.")
        return {}

    all_probs = []
    for seq in chunks:
        x = torch.tensor(seq, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            all_probs.append(F.softmax(clf(x), dim=1).numpy()[0])

    avg       = np.mean(all_probs, axis=0)
    result    = {name: float(p) for name, p in zip(ckpt["class_names"], avg)}
    predicted = max(result, key=result.get)

    print(f"Clip      : {video_path}")
    print(f"Prediction: {predicted}  ({result[predicted]:.1%})")
    print()
    for name, p in result.items():
        bar = "█" * int(p * 40)
        print(f"  {name:<15} {p:>6.1%}  {bar}")
    return result


# ── Point at a real clip to run the test ─────────────────────────────────────
# Use a littering clip from cbms-training-data, or a Screen_Clipping recording
import os as _os
from pathlib import Path as _Path
TEST_CLIP = "/kaggle/input/cbms-training-data/littering/"  # update to a specific .mp4 filename
# Alternatively test with a Screen_Clipping:
# TEST_CLIP = _os.path.join(SCREEN_CLIPPINGS_DIR, "Screen Recording 2026-04-17 113808.mp4")
if _os.path.isdir(TEST_CLIP):
    _clips = list(_Path(TEST_CLIP).glob("*.mp4"))
    TEST_CLIP = str(_clips[0]) if _clips else ""
if TEST_CLIP:
    predict_clip(TEST_CLIP, MODEL_PATH)
else:
    print("No test clip found. Add a .mp4 filename to TEST_CLIP above.")